# Probability & information lab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ankitrgupta1/silicon-to-agents/blob/main/labs/probability.ipynb)

Surprise, entropy, cross-entropy and KL divergence: the four numbers behind how language models are trained and compared. Companion to [Probability & information](../prereqs/probability.md). Runs on CPU in seconds.

In [ ]:
try:
    import s2a
except ImportError:
    %pip install -q git+https://github.com/ankitrgupta1/silicon-to-agents

import math
import numpy as np
import matplotlib.pyplot as plt
from s2a.info import entropy, cross_entropy, kl_divergence, surprise, to_bits, sample_counts
from s2a.lm import softmax

rng = np.random.default_rng(0)

## 1. Surprise

An outcome you gave probability `p` costs `-log p` when it happens. Certain things cost nothing; rare things cost a lot. In bits, each halving of the probability adds one bit.

In [ ]:
for p in [1.0, 0.5, 0.25, 0.1, 0.01, 0.001]:
    print(f"probability {p:<6} surprise {surprise(p):5.2f} nats = {to_bits(surprise(p)):5.2f} bits")

## 2. Entropy: how unpredictable is a distribution?

Entropy is the *average* surprise when outcomes come from the distribution itself. A fair coin is 1 bit; a rigged coin is less; a certain outcome is 0.

In [ ]:
heads = np.linspace(0.001, 0.999, 200)
plt.plot(heads, [to_bits(entropy([h, 1 - h])) for h in heads])
plt.xlabel("probability of heads"); plt.ylabel("entropy (bits)"); plt.show()

for name, p in [("fair 6-sided die", np.full(6, 1 / 6)), ("loaded die", [0.5, 0.1, 0.1, 0.1, 0.1, 0.1]), ("always 6", [0, 0, 0, 0, 0, 1])]:
    print(f"{name:>16}: {to_bits(entropy(p)):.2f} bits")

## 3. Cross-entropy and KL: the cost of a wrong model

If outcomes really come from `p` but you predict with `q`, your average surprise is the **cross-entropy** `H(p, q)`. It's never less than the entropy of `p`; the gap is the **KL divergence**. This is exactly what training a language model minimizes.

In [ ]:
truth = np.array([0.7, 0.2, 0.1])
for name, model in [("perfect", truth), ("close", [0.6, 0.3, 0.1]), ("uniform", np.full(3, 1 / 3)), ("overconfident, wrong", [0.05, 0.05, 0.9])]:
    ce, kl = cross_entropy(truth, model), kl_divergence(truth, model)
    print(f"{name:>20}: cross-entropy {ce:.3f} = entropy {entropy(truth):.3f} + KL {kl:.3f}")

q = np.array([0.4, 0.4, 0.2])
print(f"\nKL is not symmetric: KL(p||q) = {kl_divergence(truth, q):.4f}, KL(q||p) = {kl_divergence(q, truth):.4f}")
print("Predicting 0 for something that happens:", cross_entropy([0.5, 0.5], [1.0, 0.0]))

## 4. Learning a distribution from data

Estimate a hidden distribution by counting samples, then measure the estimate's cross-entropy against the truth. More data drives it down toward the entropy floor (with some noise from the random samples), but never below it. Training an LLM is this, with a vastly more flexible model.

In [ ]:
hidden = softmax(rng.normal(size=20) * 1.5)       # a made-up 20-token distribution
sizes = [10, 30, 100, 300, 1000, 3000, 10000]
ces = []
for n in sizes:
    counts = sample_counts(hidden, n, rng) + 0.5  # add 0.5 so nothing gets probability zero
    ces.append(cross_entropy(hidden, counts / counts.sum()))
    print(f"{n:>6} samples: cross-entropy {ces[-1]:.4f} (floor = entropy {entropy(hidden):.4f})")

plt.semilogx(sizes, ces, "o-", label="model cross-entropy")
plt.axhline(entropy(hidden), ls="--", c="gray", label="entropy (best possible)")
plt.xlabel("training samples"); plt.ylabel("nats"); plt.legend(); plt.show()

## 5. Temperature changes entropy

Dividing logits by a temperature before softmax makes the distribution sharper (below 1) or flatter (above 1).

In [ ]:
logits = np.array([2.0, 1.0, 0.5, 0.0, -1.0])
for t in [0.25, 0.5, 1.0, 2.0, 5.0]:
    p = softmax(logits / t)
    print(f"temperature {t:<4}: {np.round(p, 3)}  entropy {to_bits(entropy(p)):.2f} bits")
print(f"maximum possible for 5 options: {math.log2(5):.2f} bits")

## Try it

- Find a distribution over 4 outcomes with exactly 1.5 bits of entropy.
- In section 3, make a model that's wrong about the order of outcomes but not overconfident. How does its KL compare with the overconfident one?
- In section 4, change `+ 0.5` to `+ 0.0`. What happens with 10 samples, and why?
- A model's loss on English text is 1.0 bit per character. What does that say about how many characters it's choosing between, on average?